# Etapa 02: Preparação dos dados

### 1 - Alvo: binário (0 contra 1–4) ou as 5 classes?

Existem 28 pacientes em `num = 4`. Minha escolha será a classificação binária, pois a divisão entre pacientes doentes e não doentes é relativamente equilibrada.

O problema de manter as 5 classes é que a categoria 4 possui poucos pacientes (aproximadamente 3%), o que dificulta o treinamento. Isso significa que o modelo terá muito menos exemplos para aprender os padrões dessa classe, aumentando o risco de apresentar um desempenho ruim justamente nela.

Além disso, qualquer paciente com `num > 0` já é considerado doente, independentemente da intensidade da doença registrada.

**Decisão:** utilizar classificação binária, agrupando `num = 0` como não doente e `num = 1–4` como doente.

**Por quê:** a classificação binária apresenta uma distribuição mais equilibrada entre as classes e evita o problema de poucas observações na categoria 4.

### 2 - Zeros impossíveis: o que vira NaN e por que não substituir diretamente pela média?

Os valores `chol = 0` e `trestbps = 0` serão convertidos para `NaN`, pois representam medidas clinicamente impossíveis ou indicam informações não registradas.

Substituir esses valores pela média imediatamente pode distorcer a distribuição dos dados e introduzir informações artificiais nos registros.

Outro cuidado importante é o data leakage: se a média for calculada utilizando todo o conjunto de dados antes da separação entre treino e teste, informações do conjunto de teste acabarão influenciando o treinamento.

O mais adequado é primeiro converter os zeros para valores nulos, investigar o padrão das ausências e definir uma estratégia de imputação dentro do pipeline de treinamento.

**Decisão:** converter os zeros impossíveis em `NaN`, sem realizar imputação neste momento.

**Por quê:** primeiro é necessário compreender o padrão das ausências e evitar distorções na distribuição dos dados e vazamento de informações.

### 3 - Depois da conversão, a Suíça fica com 100% de chol faltando. Isso resolve a armadilha 2 ou só muda ela de lugar?

Na minha percepção, a armadilha apenas muda de lugar.

Anteriormente, `chol = 0` funcionava como um indicador da origem do paciente. Agora, após a conversão, `chol = NaN` pode exercer exatamente o mesmo papel.

Ou seja, o modelo ainda pode aprender que pacientes com colesterol ausente pertencem à Suíça e, consequentemente, associar essa ausência à maior prevalência de doenças observada nesse hospital.

Basicamente, só trocamos o crachá: antes era o zero, agora é o valor ausente.

**Decisão:** não considerar a conversão para `NaN` como uma solução definitiva para o problema.

**Por quê:** o padrão de ausência continua relacionado ao hospital e pode funcionar como um indicador indireto da origem dos pacientes.

### 4 - Colunas com muitos nulos (ca 66%, thal 53%, slope 34%): manter ou remover?

Muito provavelmente, eu manteria essas colunas inicialmente, pois, apesar da quantidade elevada de valores ausentes, elas contêm informações clínicas importantes que podem ajudar o modelo a identificar doenças.

**Vantagens:**
- Preservar variáveis clínicas potencialmente relevantes para a previsão.
- Aproveitar as informações disponíveis para identificar padrões associados à doença.
- Evitar descartar características que poderiam contribuir para o desempenho do modelo.

**Desvantagens:**
- A imputação pode introduzir valores artificiais e distorcer a distribuição original.
- Muitas observações precisarão de algum tratamento para os valores ausentes.
- A ausência dessas variáveis está bastante relacionada aos hospitais, então o modelo pode aprender padrões do processo de coleta em vez de características clínicas.

Um ponto que merece atenção especial é a coluna `ca`, pois ela está muito mais preenchida em Cleveland do que nos outros hospitais. Isso significa que a ausência dessa variável pode funcionar como um indicador da origem dos pacientes.

Se a ausência de `ca` estiver fortemente associada aos hospitais, o modelo poderá aprender que `ca = NaN` indica que o paciente não pertence a Cleveland, reproduzindo o mesmo problema de dependência da origem dos dados.

**Decisão:** manter inicialmente as colunas `ca`, `thal` e `slope`, mas investigar cuidadosamente os padrões de ausência antes de definir como tratá-las. Na fase 5, comparar o modelo com e sem ca/thal.

**Por quê:** essas variáveis podem conter informações clínicas importantes, mas seus valores ausentes também podem introduzir vieses relacionados aos hospitais.

### 5 - dataset e id: o que fazer com cada uma?

Minha escolha seria remover as colunas `dataset` e `id` das variáveis preditoras, mas mantê-las no DataFrame para análises e validações.

A coluna `dataset` identifica diretamente o hospital de onde o registro veio, enquanto a coluna `id` segue uma sequência de registros organizada por instituição.

Ambas podem funcionar como indicadores indiretos da origem dos pacientes e fazer com que o modelo aprenda padrões que não representam características clínicas da doença.

Além disso, um hospital novo pode não possuir uma categoria conhecida na coluna `dataset`, o que dificulta a generalização do modelo para instituições que não participaram do treinamento.

**Decisão:** remover `dataset` e `id` das variáveis utilizadas pelo modelo, mantendo-as no DataFrame para análises exploratórias e validações.

**Por quê:** essas colunas representam informações relacionadas à origem e à organização dos dados, e não características clínicas que deveriam orientar a previsão.

**Importação de bibliotecas**

In [1]:
import pandas as pd
import numpy as np

**Leitura do csv**

In [2]:
df = pd.read_csv("data/heart_disease_uci.csv")

**Criação da variável alvo**

In [3]:
df["has_disease"] = (df["num"] > 0).astype(int) # cria uma nova variavel alvo que recebe os dados da coluna "num"  

df = df.drop(columns=["num"]) # exclui "num"

**Conversão de zeros**

In [4]:
df.loc[df["chol"] == 0, "chol"] = np.nan
df.loc[df["trestbps"] == 0, "trestbps"] = np.nan

**Informações gerais**

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           920 non-null    int64  
 1   age          920 non-null    int64  
 2   sex          920 non-null    str    
 3   dataset      920 non-null    str    
 4   cp           920 non-null    str    
 5   trestbps     860 non-null    float64
 6   chol         718 non-null    float64
 7   fbs          830 non-null    object 
 8   restecg      918 non-null    str    
 9   thalch       865 non-null    float64
 10  exang        865 non-null    object 
 11  oldpeak      858 non-null    float64
 12  slope        611 non-null    str    
 13  ca           309 non-null    float64
 14  thal         434 non-null    str    
 15  has_disease  920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 115.1+ KB


**Conferir nulos por hospital**

In [6]:
colunas = ["chol", "ca", "thal", "slope"]

nulos_hospital = df.groupby("dataset")[colunas].apply(
    lambda grupo: (grupo.isna().mean() * 100).round()
)

display(nulos_hospital)

,chol,ca,thal,slope
dataset,,,,
Cleveland,0.0,2.0,1.0,0.0
Hungary,8.0,99.0,90.0,65.0
Switzerland,100.0,96.0,42.0,14.0
VA Long Beach,28.0,99.0,83.0,51.0


A minha previsão se confirmou: a ausência de chol está concentrada nos hospitais com maior proporção de pacientes doentes. Na Suíça, 100% dos valores estão ausentes, enquanto nem VA Long Beach são 28%. Isso indica que a falta de dados pode estar relacionada à origem hospitalar, e não ocorrer de forma aleatória.

**Remover colunas se necessário**

In [7]:
# df = df.drop(columns=["nome_da_coluna"]) -> manter ca, thal e slope por enquanto. Apesar de terem muitos valores ausentes, são variáveis clinicamente relevantes

**Salvar o CSV preparado**

In [8]:
df.to_csv("data/heart_preparado.csv", index=False) # index=False remove a coluna de indice do csv